# 02b — GHSL Built-Up Processing
**Step 2b of Urban Expansion vs Economic Activity pipeline**

| Step | Operation |
|------|-----------|
| 1 | Locate GHSL files, inspect one |
| 2 | Clip to metro bbox, resample to MODIS grid, binarize (≥ 1000) → `data/ghsl/{metro}/{epoch}.tif` |
| 3 | Save `data/ghsl/built_up_summary.csv` |

**Source data:** `data/raw/ghsl/` — GHS-BUILT-S R2023A, EPSG:4326, 3 arc-sec, epochs 2000–2020  
**GHSL values:** m² of built-up surface per 100 m² cell — threshold **≥ 1000** = built-up  
**No extraction needed** — reads directly from zip via `/vsizip/`

In [63]:
!pip install rasterio numpy pandas tqdm --quiet

In [64]:
import os
import math
import zipfile
import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from tqdm import tqdm

# ── Metro bounding boxes (min_lon, min_lat, max_lon, max_lat) ────────────────
BBOXES = {
    "atlanta":      (-84.55, 33.65, -84.25, 33.90),
    "austin":       (-97.94, 30.10, -97.50, 30.52),
    "charlotte":    (-81.00, 35.10, -80.70, 35.35),
    "dallas":       (-97.08, 32.62, -96.55, 33.02),
    "denver":       (-105.10, 39.60, -104.75, 39.85),
    "houston":      (-95.60, 29.65, -95.15, 29.95),
    "jacksonville": (-81.84, 30.10, -81.33, 30.54),
    "las_vegas":    (-115.35, 36.05, -115.00, 36.30),
    "nashville":    (-87.05, 35.96, -86.52, 36.35),
    "orlando":      (-81.55, 28.40, -81.20, 28.65),
    "phoenix":      (-112.32, 33.29, -111.65, 33.82),
    "raleigh":      (-78.80, 35.70, -78.50, 35.95),
    "san_antonio":  (-98.65, 29.35, -98.35, 29.55),
    "tampa":        (-82.55, 27.90, -82.35, 28.10),
}

METROS      = list(BBOXES.keys())
IMAGERY_DIR = "data/imagery"
GHSL_RAW    = "data/raw/ghsl"
GHSL_OUT    = "data/ghsl"

print(f"Metros : {len(METROS)}")
print(f"GHSL raw : {os.path.abspath(GHSL_RAW)}")

Metros : 14
GHSL raw : /Users/robintran/Desktop/APCOMP209B/209b_Final_Project/data/raw/ghsl


## Step 1 — Locate GHSL Files and Inspect

In [65]:
# ── Build epoch → /vsizip/ path ──────────────────────────────────────────────
# Always prefer the zip (guaranteed complete) over any extracted .tif
# which may be truncated from a disk-full event during extraction.

def find_ghsl_path(ghsl_raw, epoch):
    """Return rasterio-readable path for epoch. Prefers /vsizip/ over extracted tif."""
    for f in sorted(os.listdir(ghsl_raw)):
        if f.lower().endswith(".zip") and f"E{epoch}" in f:
            zpath = os.path.join(ghsl_raw, f)
            with zipfile.ZipFile(zpath) as zf:
                inner = next(
                    (n for n in zf.namelist()
                     if n.lower().endswith(".tif") and not n.lower().endswith(".ovr")),
                    None,
                )
            if inner:
                return f"/vsizip/{zpath}/{inner}"
    for f in sorted(os.listdir(ghsl_raw)):
        if (f.lower().endswith(".tif")
                and not f.lower().endswith(".ovr")
                and f"E{epoch}" in f):
            return os.path.join(ghsl_raw, f)
    return None

epoch_to_path = {}
for epoch in [2000, 2005, 2010, 2015, 2020]:
    p = find_ghsl_path(GHSL_RAW, epoch)
    if p:
        src_type = "vsizip" if p.startswith("/vsizip") else "tif"
        epoch_to_path[epoch] = p
        print(f"  {epoch} [{src_type}]  {os.path.basename(p.split('/')[-1])}")
    else:
        print(f"  {epoch}  NOT FOUND")

EPOCHS = sorted(epoch_to_path.keys())
print(f"\nEpochs: {EPOCHS}")

  2000 [vsizip]  GHS_BUILT_S_E2000_GLOBE_R2023A_4326_3ss_V1_0.tif
  2005 [vsizip]  GHS_BUILT_S_E2005_GLOBE_R2023A_4326_3ss_V1_0.tif
  2010 [vsizip]  GHS_BUILT_S_E2010_GLOBE_R2023A_4326_3ss_V1_0.tif
  2015 [vsizip]  GHS_BUILT_S_E2015_GLOBE_R2023A_4326_3ss_V1_0.tif
  2020 [vsizip]  GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0.tif

Epochs: [2000, 2005, 2010, 2015, 2020]


In [67]:
# ── Inspect first epoch — stop and review before proceeding ──────────────────
probe_epoch = EPOCHS[0]
print(f"Epoch {probe_epoch}: {os.path.basename(epoch_to_path[probe_epoch].split('/')[-1])}")
print("-" * 60)
with rasterio.open(epoch_to_path[probe_epoch]) as src:
    print(f"CRS        : {src.crs}")
    print(f"Shape      : {src.height} × {src.width}  ({src.count} band)")
    print(f"Resolution : {src.res}  degrees/pixel")
    print(f"Bounds     : {src.bounds}")
    print(f"dtype      : {src.dtypes[0]}")
    print(f"nodata     : {src.nodata}")
    # Sample over the continental US to get meaningful values
    us_win = src.window(-125, 24, -66, 50)
    sample = src.read(1, window=us_win,
                      out_shape=(256, 256), resampling=Resampling.nearest)
    nz = sample[sample > 0]
    print(f"Non-zero pixels in US sample (256×256): {len(nz)}")
    if len(nz):
        print(f"  min={nz.min()}  max={nz.max()}  mean={nz.mean():.0f}")
print("-" * 60)
print("\n⚑  CRS should be EPSG:4326. Non-zero pixels confirm data is present.")

Epoch 2000: GHS_BUILT_S_E2000_GLOBE_R2023A_4326_3ss_V1_0.tif
------------------------------------------------------------
CRS        : EPSG:4326
Shape      : 213822 × 432002  (1 band)
Resolution : (0.0008333333300326821, 0.0008333333299795073)  degrees/pixel
Bounds     : BoundingBox(left=-180.00041593133002, bottom=-89.09291610520242, right=180.0012493094487, top=89.09208317767579)
dtype      : uint16
nodata     : None
Non-zero pixels in US sample (256×256): 32844
  min=1  max=2665  mean=47
------------------------------------------------------------

⚑  CRS should be EPSG:4326. Non-zero pixels confirm data is present.


---
## Step 2 — Clip, Resample to MODIS Grid, Binarize

GHSL is already EPSG:4326 — no reprojection needed.  
Threshold: **≥ 1000** (m² built-up surface per 100 m² cell ≈ ≥10% built-up density).

In [68]:
import shutil
if os.path.exists('data/ghsl'):
    shutil.rmtree('data/ghsl')
    os.makedirs('data/ghsl')
print('Cleared data/ghsl/')

Cleared data/ghsl/


In [69]:
def pixel_area_km2(transform, center_lat_deg):
    """Area of one EPSG:4326 pixel in km² at the given latitude."""
    x_res = abs(transform.a)
    y_res = abs(transform.e)
    return (x_res * 111.320 * math.cos(math.radians(center_lat_deg))
            * y_res * 110.574)

records = []

for metro in tqdm(METROS, desc="metros", ncols=80):
    min_lon, min_lat, max_lon, max_lat = BBOXES[metro]
    center_lat = (min_lat + max_lat) / 2.0
    out_dir = os.path.join(GHSL_OUT, metro)
    os.makedirs(out_dir, exist_ok=True)

    # ── MODIS reference grid ─────────────────────────────────────────────
    modis_ref = None
    for year in range(2000, 2024):
        p = os.path.join(IMAGERY_DIR, metro, "modis_rgb", f"{year}.tif")
        if os.path.exists(p):
            with rasterio.open(p) as src:
                modis_ref = {
                    "transform": src.transform,
                    "width":     src.width,
                    "height":    src.height,
                    "crs":       src.crs,
                }
            break
    if modis_ref is None:
        print(f"[WARN] {metro}: no MODIS reference — skipping")
        continue

    for epoch in EPOCHS:
        ghsl_path = epoch_to_path[epoch]
        out_path = os.path.join(out_dir, f"{epoch}.tif")

        with rasterio.open(ghsl_path) as src:
            win = src.window(min_lon, min_lat, max_lon, max_lat)
            src_arr = src.read(
                1,
                window=win,
                out_shape=(modis_ref["height"], modis_ref["width"]),
                resampling=Resampling.nearest,
            )
            nodata = src.nodata

        if nodata is not None:
            binary = ((src_arr >= 1000) & (src_arr != nodata)).astype(np.uint8)
        else:
            binary = (src_arr >= 1000).astype(np.uint8)

        with rasterio.open(
            out_path, "w",
            driver="GTiff",
            height=modis_ref["height"], width=modis_ref["width"],
            count=1, dtype=rasterio.uint8,
            crs=modis_ref["crs"], transform=modis_ref["transform"],
            compress="lzw", nodata=255,
        ) as dst:
            dst.write(binary, 1)

        # built_up_km2 from native GHSL resolution clipped to metro bbox only
        with rasterio.open(epoch_to_path[epoch]) as src:
            metro_win = src.window(min_lon, min_lat, max_lon, max_lat)
            metro_arr = src.read(1, window=metro_win)
            native_px_area = pixel_area_km2(src.transform, center_lat)
        metro_binary = (metro_arr >= 1000).astype(np.uint8)
        built_km2 = float(metro_binary.sum()) * native_px_area
        records.append({"metro": metro, "epoch": epoch, "built_up_km2": built_km2})
        print(f"  {metro}/{epoch}.tif  built_up={built_km2:.1f} km²  "
              f"pixels={int(metro_binary.sum())}")

print("\n✓ Step 2 complete.")

metros:   0%|                                            | 0/14 [00:00<?, ?it/s]

  atlanta/2000.tif  built_up=248.4 km²  pixels=34966
  atlanta/2005.tif  built_up=274.6 km²  pixels=38648
  atlanta/2010.tif  built_up=311.2 km²  pixels=43793
  atlanta/2015.tif  built_up=341.6 km²  pixels=48070


metros:   7%|██▌                                 | 1/14 [01:20<17:32, 80.94s/it]

  atlanta/2020.tif  built_up=349.3 km²  pixels=49160
  austin/2000.tif  built_up=341.4 km²  pixels=46267
  austin/2005.tif  built_up=394.1 km²  pixels=53404
  austin/2010.tif  built_up=468.5 km²  pixels=63493
  austin/2015.tif  built_up=515.7 km²  pixels=69887


metros:  14%|█████▏                              | 2/14 [02:58<18:08, 90.70s/it]

...

  raleigh/2020.tif  built_up=246.3 km²  pixels=35541
  san_antonio/2000.tif  built_up=348.5 km²  pixels=46824
  san_antonio/2005.tif  built_up=367.4 km²  pixels=49362
  san_antonio/2010.tif  built_up=387.1 km²  pixels=52004
  san_antonio/2015.tif  built_up=401.5 km²  pixels=53938


metros:  93%|████████████████████████████████▌  | 13/14 [18:38<01:26, 86.77s/it]

  san_antonio/2020.tif  built_up=413.2 km²  pixels=55511
  tampa/2000.tif  built_up=251.9 km²  pixels=33377
  tampa/2005.tif  built_up=260.0 km²  pixels=34446
  tampa/2010.tif  built_up=267.4 km²  pixels=35423
  tampa/2015.tif  built_up=272.5 km²  pixels=36107


metros: 100%|███████████████████████████████████| 14/14 [20:20<00:00, 87.15s/it]

  tampa/2020.tif  built_up=275.0 km²  pixels=36434

✓ Step 2 complete.


## Step 3 — Save built_up_summary.csv

In [70]:
summary = (pd.DataFrame(records)
             .sort_values(["metro", "epoch"])
             .reset_index(drop=True))

csv_path = os.path.join(GHSL_OUT, "built_up_summary.csv")
summary.to_csv(csv_path, index=False)

print(f"Saved: {csv_path}  ({len(summary)} rows)")
print()
print(summary.to_string(index=False))

Saved: data/ghsl/built_up_summary.csv  (70 rows)

       metro  epoch  built_up_km2
     atlanta   2000    248.444496
     atlanta   2005    274.606271
     atlanta   2010    311.163125
     atlanta   2015    341.552563
     atlanta   2020    349.297358
      austin   2000    341.429127
      austin   2005    394.096896
      austin   2010    468.549064
      austin   2015    515.733835
      austin   2020    549.244322
   charlotte   2000    193.723578
   charlotte   2005    217.779293
   charlotte   2010    254.089808
   charlotte   2015    285.226074
   charlotte   2020    312.682409
      dallas   2000   1055.957866
      dallas   2005   1104.188110
      dallas   2010   1161.038594
      dallas   2015   1200.770717
      dallas   2020   1229.490484
      denver   2000    492.661662
      denver   2005    505.882845
      denver   2010    521.766674
      denver   2015    534.185777
      denver   2020    546.802112
     houston   2000    804.562346
     houston   2005    840.65661